# Reconciliation 360 Analysis

This notebook explores the Entity 360 and Reconciliation Fact data models created with dbt.

**Data Sources:**
- `DBAPI_REPLICA_DB.PUBLIC_MARTS.ENTITY_360` - Entity-level reconciliation metrics
- `DBAPI_REPLICA_DB.PUBLIC_MARTS.RECONCILIATION_FACT` - Detailed reconciliation records

**Semantic Views (for Cortex Agent):**
- `DBAPI_REPLICA_DB.PUBLIC.SV_ENTITY_360`
- `DBAPI_REPLICA_DB.PUBLIC.SV_RECONCILIATION_FACT`

In [ ]:
%%sql -r df_overview
-- Overview: Row counts and date ranges
SELECT 
    'entity_360' AS table_name, 
    COUNT(*) AS row_count,
    MIN(period_end_date) AS earliest_period,
    MAX(period_end_date) AS latest_period
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.ENTITY_360
UNION ALL
SELECT 
    'reconciliation_fact', 
    COUNT(*),
    MIN(period_end_date),
    MAX(period_end_date)
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.RECONCILIATION_FACT

## Risk Distribution
How many entity-periods fall into each variance risk category?

In [ ]:
%%sql -r df_risk
-- Risk level distribution
SELECT 
    variance_risk_level,
    COUNT(*) AS entity_period_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_total,
    SUM(total_absolute_variance) AS total_variance_amount
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.ENTITY_360
GROUP BY variance_risk_level
ORDER BY entity_period_count DESC

## High Risk Entities
Which entities have the highest variance risk?

In [ ]:
%%sql -r df_high_risk
-- Top 15 high risk entities
SELECT 
    entity_name,
    entity_code,
    period_end_date,
    total_assignments,
    variance_count,
    total_variance,
    total_absolute_variance,
    max_single_variance,
    variance_risk_level
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.ENTITY_360
WHERE variance_risk_level = 'High Risk'
ORDER BY total_absolute_variance DESC
LIMIT 15

## Reconciliation Status Breakdown
What is the status of reconciliations across all records?

In [ ]:
%%sql -r df_status
-- Reconciliation status summary
SELECT 
    reconciliation_status,
    COUNT(*) AS recon_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_total,
    SUM(balance_bank) AS total_bank_balance,
    SUM(ABS(bank_calc_diff)) AS total_difference
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.RECONCILIATION_FACT
GROUP BY reconciliation_status
ORDER BY recon_count DESC

## Quarterly Trend Analysis
How are reconciliations and variances trending over time?

In [ ]:
%%sql -r df_quarterly
-- Quarterly summary
SELECT 
    period_year,
    period_quarter,
    COUNT(DISTINCT entity_id) AS entities,
    SUM(total_assignments) AS total_assignments,
    SUM(total_gl_balance) AS sum_gl_balance,
    SUM(total_variance) AS sum_variance,
    SUM(CASE WHEN variance_risk_level = 'High Risk' THEN 1 ELSE 0 END) AS high_risk_count
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.ENTITY_360
GROUP BY period_year, period_quarter
ORDER BY period_year, period_quarter

## Largest Unreconciled Differences
Which reconciliations need the most attention?

In [ ]:
%%sql -r df_needs_review
-- Top 20 items needing review
SELECT 
    entity_name,
    account_combination,
    period_end_date,
    currency,
    balance_bank,
    balance_calculated,
    bank_calc_diff,
    reconciliation_status
FROM DBAPI_REPLICA_DB.PUBLIC_MARTS.RECONCILIATION_FACT
WHERE reconciliation_status = 'Needs Review'
ORDER BY ABS(bank_calc_diff) DESC
LIMIT 20